# Sentiment Pipeline NLP - YouTube Comment: Benchmark & Evaluation
Project Capstone: Rifki Afina Putri, S.T., M.S., Ph.D.

Menilai **nada/emosi (sentiment)** komentar YouTube tentang PLTU/PLTN: **Positive / Neutral / Negative**. Hanya komentar yang sudah relevan yang dinilai (Pedoman A1). Sentimen menilai nada, BUKAN arah dukungan (Pedoman C1).

* **Input File Tunggal**: `stance_sentiment_input_yt.csv` (99 komentar relevan + konteks video).
* **Model**: Google AI Studio API (`gemma-4-26b-a4b-it`), key dari **Colab Secrets** (`GEMMA`).
* **Label (3 kelas)**: `Positive`, `Neutral`, `Negative`. Baris dengan label kosong di-drop.
* **3 Konfigurasi**: `majority` (tanpa API, lantai baseline), `zero_shot` (prompt ringkas tanpa pedoman), `v2_pedoman` (prompt berbasis Pedoman Anotasi v2 bagian C (Sentimen) + konteks video).
* **Stateful & Resumable**: progres tiap config disimpan ke `*_checkpoint.csv`. Kalau kuota API habis atau Colab terputus, jalankan ulang -> otomatis lanjut dari komentar yang belum selesai.
* **Metrik 3 kelas**: confusion matrix, akurasi, macro-F1, precision/recall/F1 per kelas (sklearn).

## 1. Import Libraries & Muat Dataset Input
Upload `stance_sentiment_input_yt.csv` ke Colab, lalu jalankan sel ini.

In [1]:
import pandas as pd
import numpy as np
import os, re, json, time

INPUT_FILE = 'stance_sentiment_input_yt.csv'
LABEL_COL = 'sentiment'            # kolom ground-truth untuk notebook ini
LABELS = ['Positive', 'Neutral', 'Negative']  # 3 kelas yang dinilai

if not os.path.exists(INPUT_FILE):
    raise FileNotFoundError(f"File '{INPUT_FILE}' tidak ada. Upload dulu ke Colab (ikon folder di kiri).")

df = pd.read_csv(INPUT_FILE, dtype=str, keep_default_na=False)
df.columns = df.columns.astype(str).str.strip()
print(f'[ok] {len(df)} komentar relevan dimuat | kolom: {df.columns.tolist()}')

# Hanya baris dengan label valid untuk tugas ini yang dievaluasi (label kosong di-drop).
before = len(df)
df = df[df[LABEL_COL].isin(LABELS)].reset_index(drop=True)
df['i'] = df.index  # id stabil untuk checkpoint
print(f'[ok] {len(df)}/{before} komentar punya label {LABEL_COL} valid | distribusi:')
print(df[LABEL_COL].value_counts().to_dict())
df[['i', 'target', LABEL_COL, 'text']].head(5)

[ok] 99 komentar relevan dimuat | kolom: ['no', 'cid', 'video_id', 'video_title', 'keyword', 'target', 'language', 'is_reply', 'text', 'stance', 'sentiment', 'video_channel', 'video_desc', 'video_tags']
[ok] 91/99 komentar punya label sentiment valid | distribusi:
{'Negative': 40, 'Neutral': 26, 'Positive': 25}


,i,target,sentiment,text
0,0,PLTU,Neutral,Tapi harga buy PLN/ BB dalam negeri secara atu...
1,1,PLTU,Neutral,PLTU Suralaya juga dong berarti 2025 resmi ga ...
2,2,PLTU,Neutral,"nyungsep2 asbun aja om benix, listrik lu noh 7..."
3,3,UMUM,Negative,"Negara banyak utang ke pln, pln banyak utsng j..."
4,4,UMUM,Negative,Nah itu yg kami di pulau Sumatra rasakan


## 2. Set Up Google GenAI Client & API Key
API key diambil dari **Colab Secrets** bernama `GEMMA` (ikon kunci di kiri). Jangan menempel key langsung di kode.

In [2]:
from google import genai
from google.genai import types

api_key = None
try:
    from google.colab import userdata
    api_key = userdata.get('GEMMA') or userdata.get('GEMINI_API_KEY')
except Exception:
    api_key = os.environ.get('GEMMA') or os.environ.get('GEMINI_API_KEY')

if not api_key:
    raise RuntimeError('API key tidak ditemukan. Tambahkan Secret bernama GEMMA di Colab (ikon kunci).')

MODEL = 'gemma-4-26b-a4b-it'
client = genai.Client(api_key=api_key)
print('Google GenAI Client siap. Model:', MODEL)

Google GenAI Client siap. Model: gemma-4-26b-a4b-it


## 3. Prompt (inline) & Parser JSON Toleran
* `PROMPT_ZERO_SHOT`: ringkas, tanpa pedoman -> baseline.
* `PROMPT_V2`: berbasis Pedoman Anotasi v2 bagian C (Sentimen) + konteks video. **Boleh diedit langsung di sini.**
* Komentar dikelompokkan per video agar judul/deskripsi cukup ditulis sekali; target menempel di tiap komentar.
* Parser toleran terhadap pagar ```json, teks pembuka, dan karakter sisa di belakang JSON.

In [3]:
PROMPT_ZERO_SHOT = 'Kamu menilai SENTIMENT (nada/emosi) komentar YouTube tentang PLTU (pembangkit batu bara) atau PLTN (pembangkit nuklir) di Indonesia.\nSemua komentar di bawah sudah dipastikan relevan. Untuk SETIAP komentar, tentukan sentimen bagian yang membahas PLTU/PLTN. Nilai tiap komentar sendiri-sendiri.\n\nSENTIMENT (pilih satu):\n- Positive: pujian, harapan, optimis, bangga, menyebut manfaat (murah, bersih, aman, maju).\n- Negative: takut, marah, kecewa, kesal, sinis, sarkas, mengejek, pesimis, menyebut bahaya/kerugian.\n- Neutral: pertanyaan tulus, fakta, info, analisis datar, tanpa kata penilaian.\n\n{{ITEMS}}\n\n== FORMAT JAWABAN ==\nBalas HANYA dengan JSON array (tanpa teks lain, tanpa ```), satu objek per komentar, urutan sama dengan input:\n[{"id": <id>, "alasan": "<maksimal 10 kata>", "sentiment": "Positive|Neutral|Negative"}]\n'

PROMPT_V2 = 'Kamu adalah annotator untuk riset opini publik tentang PLTU (pembangkit listrik batu bara) dan PLTN (pembangkit listrik tenaga nuklir) di Indonesia.\nSemua komentar di bawah SUDAH dipastikan relevan. Untuk SETIAP komentar, tentukan SENTIMENT menurut Pedoman Anotasi v2 berikut. Nilai tiap komentar sendiri-sendiri.\n\n== PRINSIP (A) ==\nA3. Yang dinilai isi komentar. Judul/deskripsi video hanya konteks terbatas. Jangan menilai dari judul saja.\nA4. Ragu di antara dua label -> pilih yang lebih netral.\n\n== SENTIMENT (Positive / Neutral / Negative) ==\nC1. Sentimen = nada/emosi bagian komentar yang membahas PLTU/PLTN Indonesia, BUKAN arah dukungannya (stance). Nada dan sikap boleh beda arah.\nC2. Positive: pujian, harapan, bangga, optimis, antusias, yakin aman, menyebut manfaat (murah, bersih, aman, maju, efisien).\nC3. Neutral: pertanyaan tulus, fakta, info, analisis datar, koreksi data, usulan tanpa kata penilaian, dukungan/penolakan yang diucapkan datar ("Belum saatnya", "Mending indo jangan punya PLTN deh").\nC4. Negative: takut, marah, kecewa, kesal, iba/prihatin, sinis, sarkas, mengejek, pesimis, umpatan, menyebut bahaya atau kerugian.\nC5. Abaikan ejekan ke komentator lain atau ke kreator, salam, dan terima kasih ke kreator. Ejekan ke kebijakan, pemerintah, atau Indonesia sebagai pelaksana program tetap dihitung.\nC6. Kalau ada "tapi/namun/padahal", ambil bagian SESUDAHNYA sebagai kesimpulan. Neutral karena seimbang hanya dipakai kalau tidak ada penanda kesimpulan sama sekali.\nC7. Penjelasan panjang dinilai dari kesimpulannya: diakhiri nada yakin/positif -> Positive, diakhiri kekesalan -> Negative.\nC8. Sarkasme dinilai dari maksudnya. Pertanyaan retoris yang mengejek/menantang/menyindir ("Emang bisaa...", "trs solusinya apa?") -> Negative. Tanpa penanda sarkasme, baca sesuai bunyinya.\nC9. Kritik bernada menuntut/kesal ("Bayar dulu dong") -> Negative. Kritik berupa analisis datar ("Berarti PLN tidak sanggup membayar...") -> Neutral.\nC10. Iba atau prihatin tetap Negative ("Kasian karyawan pltu nya").\nC11. Usulan dinilai dari kata-katanya: ada kata manfaat/pujian ("lbh ramah lingkungan") -> Positive; tanpa kata penilaian -> Neutral.\n\n== CONTOH ==\n"Semoga PLTN cepat terwujud, Indonesia maju" -> Positive (C2)\n"Ngeri kalau meledak kayak Chernobyl" -> Negative (C4)\n"Berarti PLN tidak sanggup membayar pasokan batu bara" -> Neutral (C9)\n"Mending indo jangan punya PLTN deh" -> Neutral (C3, penolakan datar)\n"Emang bisaa negara kita bikin PLTN? 1000 tahun lagi kali" -> Negative (C8)\n"Kasian karyawan PLTU yang kena pensiun dini" -> Negative (C10)\n\n{{ITEMS}}\n\n== FORMAT JAWABAN ==\nBalas HANYA dengan JSON array (tanpa teks lain, tanpa ```), satu objek per komentar, urutan sama dengan input:\n[{"id": <id>, "alasan": "<maksimal 12 kata, sebut kode aturan>", "sentiment": "Positive|Neutral|Negative"}]\n'

def render_items(batch, context):
    """context='title' -> judul saja; context='full' -> + channel/deskripsi/tag."""
    by_video = {}
    for it in batch:
        by_video.setdefault(it['video_id'], []).append(it)
    out = ['== VIDEO DAN KOMENTAR ==']
    for n, group in enumerate(by_video.values(), 1):
        g0 = group[0]
        out.append(f'\nVideo {n}')
        out.append(f"Judul: {g0['video_title']}")
        if context == 'full':
            if g0['video_channel']: out.append(f"Channel: {g0['video_channel']}")
            if g0['video_desc']:    out.append(f"Deskripsi: {g0['video_desc']}")
            if g0['video_tags']:    out.append(f"Tag: {g0['video_tags']}")
        out.append('Komentar:')
        for it in group:
            jenis = 'reply' if str(it['is_reply']) == '1' else 'komentar'
            out.append(f"[{it['i']}] (target: {it['target']}) ({jenis}) {it['text']}")
    return '\n'.join(out)

def build_prompt(template, batch, context):
    return template.replace('{{ITEMS}}', render_items(batch, context))

FENCE_RE = re.compile(r'```(?:json)?', re.I)
_MAP = {
    "positive": "Positive", "positif": "Positive",
    "negative": "Negative", "negatif": "Negative",
    "neutral": "Neutral", "netral": "Neutral",
}

def _norm(v):
    return _MAP.get(v.strip().lower()) if isinstance(v, str) else None

def parse_response(text, expected):
    """{i: (label, alasan)} untuk id yang diminta; id dengan label tak dikenal diabaikan."""
    if not text: return {}
    tx = FENCE_RE.sub('', text)
    dec = json.JSONDecoder()
    for m in re.finditer(r'[\[{]', tx):
        try:
            data, _ = dec.raw_decode(tx[m.start():])
        except json.JSONDecodeError:
            continue
        if isinstance(data, dict):
            data = data.get('results', data.get('hasil', [data]))
        if not isinstance(data, list):
            continue
        out = {}
        for it in data:
            if not isinstance(it, dict): continue
            try: i = int(it.get('id'))
            except (TypeError, ValueError): continue
            lab = _norm(it.get('sentiment', it.get('label')))
            if i in expected and lab is not None:
                out[i] = (lab, str(it.get('alasan', ''))[:160])
        if out:
            return out
    return {}

print('Prompt & parser siap. Panjang PROMPT_V2:', len(PROMPT_V2), 'karakter')

Prompt & parser siap. Panjang PROMPT_V2: 2823 karakter


## 4. Eksekusi Stateful & Resumable (3 Konfigurasi)
* `majority` tidak memanggil API (tebak kelas mayoritas gold).
* `zero_shot` & `v2_pedoman` memanggil API dengan pacing + retry (500/429 di-retry, gagal total = `unknown`, bukan salah).
* Checkpoint per config: `preds_<task>_<config>.csv`. Jalankan ulang sel ini untuk melanjutkan yang terputus.

In [4]:
BATCH_SIZE = 8
MIN_GAP = 4.0        # jeda antar panggilan (detik) -> aman untuk free tier
MAX_RETRIES = 6

CONFIGS = {
    'majority':   {'kind': 'majority'},
    'zero_shot':  {'template': PROMPT_ZERO_SHOT, 'context': 'title'},
    'v2_pedoman': {'template': PROMPT_V2,        'context': 'full'},
}

ITEMS = df.to_dict('records')
EXPECTED_IDS = [int(it['i']) for it in ITEMS]

def make_batches(items, size):
    seq = sorted(items, key=lambda x: (x['video_id'], int(x['i'])))
    return [seq[k:k+size] for k in range(0, len(seq), size)]

def generate(prompt):
    cfg = types.GenerateContentConfig(temperature=0.0, response_mime_type='application/json')
    resp = client.models.generate_content(model=MODEL, contents=prompt, config=cfg)
    return resp.text or ''

def backoff(err, attempt):
    s = str(err)
    if re.search(r'429|RESOURCE_EXHAUSTED|quota', s, re.I): return 30 + 10 * attempt
    return min(5 * (2 ** attempt), 60)

def run_api_config(name, cfg):
    path = f'preds_{LABEL_COL}_{name}.csv'
    preds = {}
    if os.path.exists(path):
        prev = pd.read_csv(path, dtype=str, keep_default_na=False)
        preds = {int(r.i): (r.pred, r.alasan) for r in prev.itertuples(index=False) if r.pred in LABELS}
        print(f'[resume] {name}: {len(preds)} sudah ada')
    todo = [it for it in ITEMS if int(it['i']) not in preds]
    batches = make_batches(todo, BATCH_SIZE)
    print(f'[{name}] {len(ITEMS)} komentar | sisa {len(todo)} | {len(batches)} panggilan API')
    last = 0.0
    for bn, batch in enumerate(batches, 1):
        pending, attempt = list(batch), 0
        while pending and attempt <= MAX_RETRIES:
            gap = MIN_GAP - (time.time() - last)
            if gap > 0: time.sleep(gap)
            last = time.time()
            prompt = build_prompt(cfg['template'], pending, cfg['context'])
            try:
                text = generate(prompt)
            except Exception as e:
                msg = str(e)
                if re.search(r'API key|API_KEY_INVALID|UNAUTHENTICATED|PERMISSION_DENIED|\b401\b|\b403\b', msg):
                    raise RuntimeError('API key ditolak/tak berizin: ' + msg[:200])
                w = backoff(e, attempt)
                print(f'    ! {name} b{bn}: {msg[:80]} -> ulang {w:.0f}s ({attempt+1}/{MAX_RETRIES})')
                time.sleep(w); attempt += 1; continue
            parsed = parse_response(text, {int(it['i']) for it in pending})
            if not parsed:
                w = backoff(RuntimeError(''), attempt)
                print(f'    ! {name} b{bn}: jawaban tak terbaca -> ulang {w:.0f}s ({attempt+1}/{MAX_RETRIES})')
                time.sleep(w); attempt += 1; continue
            preds.update(parsed)
            pending = [it for it in pending if int(it['i']) not in parsed]
            if pending: attempt += 1
        # simpan checkpoint setelah tiap batch
        rows = [{'i': int(it['i']), 'cid': it['cid'],
                 'pred': preds[int(it['i'])][0] if int(it['i']) in preds else '',
                 'alasan': preds.get(int(it['i']), ('', ''))[1]} for it in ITEMS]
        pd.DataFrame(rows).to_csv(path, index=False, encoding='utf-8')
        print(f'  batch {bn}/{len(batches)} selesai ({len(preds)}/{len(ITEMS)} terisi)')
    return preds

def run_majority():
    from collections import Counter
    top = Counter(df[LABEL_COL]).most_common(1)[0][0]
    return {int(it['i']): (top, f'majority={top}') for it in ITEMS}

ALL_PREDS = {}
for name, cfg in CONFIGS.items():
    ALL_PREDS[name] = run_majority() if cfg.get('kind') == 'majority' else run_api_config(name, cfg)
print('\nSemua konfigurasi selesai.')

[zero_shot] 91 komentar | sisa 91 | 12 panggilan API
    ! zero_shot b1: 500 INTERNAL. {'error': {'code': 500, 'message': 'Internal error encountered.',  -> ulang 5s (1/6)
    ! zero_shot b1: 500 INTERNAL. {'error': {'code': 500, 'message': 'Internal error encountered.',  -> ulang 10s (2/6)
    ! zero_shot b1: 500 INTERNAL. {'error': {'code': 500, 'message': 'Internal error encountered.',  -> ulang 20s (3/6)
  batch 1/12 selesai (8/91 terisi)
  batch 2/12 selesai (16/91 terisi)
  batch 3/12 selesai (24/91 terisi)
  batch 4/12 selesai (32/91 terisi)
  batch 5/12 selesai (40/91 terisi)
  batch 6/12 selesai (48/91 terisi)
  batch 7/12 selesai (56/91 terisi)
  batch 8/12 selesai (64/91 terisi)
    ! zero_shot b9: 500 INTERNAL. {'error': {'code': 500, 'message': 'Internal error encountered.',  -> ulang 5s (1/6)
  batch 9/12 selesai (72/91 terisi)
  batch 10/12 selesai (80/91 terisi)
  batch 11/12 selesai (88/91 terisi)
  batch 12/12 selesai (91/91 terisi)
[v2_pedoman] 91 komentar | sisa 91 

## 5. Evaluasi Metrik 3 Kelas & Analisis Error
* **Catatan**: kelas tidak seimbang, jadi **macro-F1** dan F1 per-kelas lebih penting dari akurasi mentah.
* `unknown` = komentar yang gagal diproses setelah semua retry; **tidak** dihitung sebagai salah.
* File `errors_<task>_<config>.csv` berisi komentar yang salah (prediksi vs gold + alasan model) untuk audit annotator.

In [5]:
from sklearn.metrics import confusion_matrix, classification_report, f1_score, accuracy_score

gold = {int(it['i']): it[LABEL_COL] for it in ITEMS}
summary = []
for name, preds in ALL_PREDS.items():
    ids = [i for i in EXPECTED_IDS if i in preds]      # hanya yang terjawab
    unknown = len(EXPECTED_IDS) - len(ids)
    y_true = [gold[i] for i in ids]
    y_pred = [preds[i][0] for i in ids]
    acc = accuracy_score(y_true, y_pred) if ids else float('nan')
    macro = f1_score(y_true, y_pred, labels=LABELS, average='macro', zero_division=0) if ids else float('nan')
    per = f1_score(y_true, y_pred, labels=LABELS, average=None, zero_division=0) if ids else [float('nan')]*3
    row = {'config': name, 'n': len(ids), 'unknown': unknown, 'accuracy': acc, 'macro_F1': macro}
    row.update({f'{lab}_F1': per[k] for k, lab in enumerate(LABELS)})
    summary.append(row)

summary_df = pd.DataFrame(summary)
pct = ['accuracy', 'macro_F1'] + [f'{lab}_F1' for lab in LABELS]
print('=' * 90)
print('HASIL SENTIMENT pada', len(EXPECTED_IDS), 'komentar relevan | model', MODEL)
print('=' * 90)
display(summary_df.style.format({c: '{:.1%}' for c in pct}))

# rincian per config: confusion matrix + classification_report + dump error
for name, preds in ALL_PREDS.items():
    print(f'\n--- {name} ---')
    ids = [i for i in EXPECTED_IDS if i in preds]
    if not ids:
        print('  (semua komentar unknown / tak ada prediksi terbaca — lewati metrik config ini)')
        continue
    y_true = [gold[i] for i in ids]; y_pred = [preds[i][0] for i in ids]
    cm = confusion_matrix(y_true, y_pred, labels=LABELS)
    print('confusion matrix (baris=gold, kolom=pred):')
    print(pd.DataFrame(cm, index=[f'gold_{l}' for l in LABELS], columns=[f'pred_{l}' for l in LABELS]))
    print(classification_report(y_true, y_pred, labels=LABELS, zero_division=0))
    byid = {it['i']: it for it in ITEMS}
    wrong = [{'cid': byid[i]['cid'], 'target': byid[i]['target'], 'text': byid[i]['text'],
              'gold': gold[i], 'pred': preds[i][0], 'alasan': preds[i][1]}
             for i in ids if preds[i][0] != gold[i]]
    pd.DataFrame(wrong).to_csv(f'errors_{LABEL_COL}_{name}.csv', index=False, encoding='utf-8')
    print(f'  {len(wrong)} salah -> errors_{LABEL_COL}_{name}.csv')

summary_df.to_csv(f'summary_{LABEL_COL}_latest.csv', index=False, encoding='utf-8')
print('\nRingkasan disimpan: summary_' + LABEL_COL + '_latest.csv')

HASIL SENTIMENT pada 91 komentar relevan | model gemma-4-26b-a4b-it


,config,n,unknown,accuracy,macro_F1,Positive_F1,Neutral_F1,Negative_F1
0,majority,91,0,44.0%,20.4%,0.0%,0.0%,61.1%
1,zero_shot,91,0,78.0%,77.7%,80.0%,71.2%,82.1%
2,v2_pedoman,91,0,80.2%,79.8%,80.9%,72.1%,86.5%



--- majority ---
confusion matrix (baris=gold, kolom=pred):
               pred_Positive  pred_Neutral  pred_Negative
gold_Positive              0             0             25
gold_Neutral               0             0             26
gold_Negative              0             0             40
              precision    recall  f1-score   support

    Positive       0.00      0.00      0.00        25
     Neutral       0.00      0.00      0.00        26
    Negative       0.44      1.00      0.61        40

    accuracy                           0.44        91
   macro avg       0.15      0.33      0.20        91
weighted avg       0.19      0.44      0.27        91

  51 salah -> errors_sentiment_majority.csv

--- zero_shot ---
confusion matrix (baris=gold, kolom=pred):
               pred_Positive  pred_Neutral  pred_Negative
gold_Positive             18             4              3
gold_Neutral               2            21              3
gold_Negative              0             8    

## 6. Flowchart Metodologi Sentiment (Mermaid Diagram)

In [6]:
import base64
from IPython.display import Image, display

def render_mermaid(graph):
    b64 = base64.b64encode(graph.encode('ascii')).decode('ascii')
    display(Image(url='https://mermaid.ink/img/' + b64))

render_mermaid('''
graph TD
    A[Muat stance_sentiment_input_yt.csv<br/>99 komentar relevan]:::s
    B[Filter label sentiment valid<br/>drop label kosong]:::s
    C{Konfigurasi}:::f
    M[majority: tebak kelas mayoritas<br/>0 panggilan API]:::p
    Z[zero_shot: prompt ringkas<br/>judul saja]:::a
    V[v2_pedoman: prompt Pedoman<br/>+ konteks video + target]:::a
    R[Batch per video + pacing + retry<br/>checkpoint resumable]:::p
    E[Metrik 3 kelas: confusion matrix,<br/>macro-F1, report per kelas]:::v
    A --> B --> C
    C --> M --> E
    C --> Z --> R
    C --> V --> R --> E
    classDef s fill:#2b2d42,color:#fff,stroke:#1d1e2c,rx:6px;
    classDef f fill:#f39c12,color:#fff,stroke:#d68910,rx:6px;
    classDef p fill:#457b9d,color:#fff,stroke:#1d3557,rx:6px;
    classDef a fill:#3a86ff,color:#fff,stroke:#004e92,rx:6px;
    classDef v fill:#06d6a0,color:#fff,stroke:#0f8b6d,rx:6px;
''')